# 17 - D22: train every architecture on `train_d22.csv` with class weights

**In short:** the six architectures of notebook 09 are trained again on `train_d22.csv`
(`train_d17.csv` plus the South African maize field photos from notebook 16), with the same
recipe as before and one change: **balanced class weights** in the loss. This notebook only
trains and scores; notebook 18 decides, on the CPU, with rules fixed before training.

**Why class weights (decided 22 Oct 2026, before training):** the rule was "largest class
over smallest class at most 3: no weighting; over 3: balanced weights, capped at 5". In
`train_d22.csv` potato late blight has 2,904 photos and tomato early blight 719, a ratio of
4.0, so each class's loss is weighted by total / (classes x class photos): from 0.52 (potato
late blight) to 2.1 (tomato early blight). The cap of 5 is not reached. The weights are
checked below against `results/d22_maize_data/class_balance.csv`.

**Order:** ResNet-50 first, with seeds 42, 43 and 44 (the "v3" candidate that notebook 18
compares with v2 by the D17 rule, 3 seeds against 3). Then one run each of EfficientNet-B0,
MobileNetV3, DenseNet-121, ConvNeXt-Tiny (at 1e-3 and at 1e-4, as in notebook 09) and
Swin-T, for the architecture check.

**Nothing earlier changes:** v1 (`compare_resnet50`), v2 (`d17_resnet50_seed44`) and every
other earlier run are only read. New runs go into new folders (`d22_<architecture>...`,
`d22_comparison`).

**How to run:** T4 GPU, in two passes.

1. *Run all* as it is (`TRAIN_OTHERS = False`): the three ResNet runs, about 30 minutes each,
   then the results are pushed to `colab/d22-training-resnet`. Notebook 18 can start on
   these while the others train.
2. Set `TRAIN_OTHERS = True` in the settings cell and *Run all* again: the ResNet runs are
   skipped, the other six are trained (ConvNeXt and Swin take longer), and everything is
   pushed to `colab/d22-training-all`.
 Progress is saved every epoch: after a disconnect, *Run all* again continues the
unfinished run and skips finished ones. `last.pt` is deleted when a run finishes, to keep
Drive space free. Cut-off: whatever has finished by **Sunday 25 Oct, 12:00** is compared.

In [13]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
## main...origin/main
 M ml/notebooks/17_d22_training.ipynb
 M ml/results/d22_training/notebook_resnet.ipynb
?? ml/results/fusion_quick_test/


In [15]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings

`RECIPE` is v1's and v2's (notebook 09's shared settings), checked against v2's saved
`settings.json`. The run list is fixed; only `TRAIN_OTHERS` may be set to `False` to stop
after the three ResNet runs: `False` for the first pass, `True` for the second.

In [16]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd
import torch

from src import calibration as cal, metrics, training
from src.data import image_dataset
for module in (training, metrics, cal, image_dataset):
    importlib.reload(module)

TRAIN_OTHERS = True   # first pass: ResNet only; set True for the second pass

RECIPE = {"image_size": 224, "batch_size": 64, "epochs": 15, "patience": 4, "weight_decay": 1e-2}
CAP = 5.0
WEIGHT_RULE = "balanced (total / (classes x class photos)), capped at 5; decided 22 Oct 2026 because largest/smallest = 4.0 > 3"

# run name: (architecture, seed, learning rate). ResNet first.
RESNET_RUNS = {f"d22_resnet50_seed{s}": ("resnet50", s, 1e-3) for s in (42, 43, 44)}
OTHER_RUNS = {
    "d22_efficientnet_b0": ("efficientnet_b0", 42, 1e-3),
    "d22_mobilenet_v3_large": ("mobilenet_v3_large", 42, 1e-3),
    "d22_densenet121": ("densenet121", 42, 1e-3),
    "d22_convnext_tiny": ("convnext_tiny", 42, 1e-3),
    "d22_convnext_tiny_lr1e-04": ("convnext_tiny", 42, 1e-4),
    "d22_swin_t": ("swin_t", 42, 1e-3),
}
RUN_PLAN = {**RESNET_RUNS, **(OTHER_RUNS if TRAIN_OTHERS else {})}

V2_RUN = "d17_resnet50_seed44"                         # v2, deployed: read only
V2_SCORES = config.MODELS_DIR / "d17_comparison"        # v2's saved scores (notebook 15)
OUT_DIR = config.MODELS_DIR / "d22_comparison"

v2_settings = json.loads((config.MODELS_DIR / V2_RUN / "settings.json").read_text())
for key, value in RECIPE.items():
    assert v2_settings[key] == value, f"{key}: v2 used {v2_settings[key]}, recipe says {value}"
EARLIER = {p.name for p in config.MODELS_DIR.iterdir() if p.is_dir() and not p.name.startswith("d22_")}
for name in [*RUN_PLAN, OUT_DIR.name]:
    assert name.startswith("d22_") and name not in EARLIER, f"{name} would overwrite an earlier run"

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(len(RUN_PLAN), "runs planned | recipe confirmed against v2 | GPU:", torch.cuda.get_device_name(0))

9 runs planned | recipe confirmed against v2 | GPU: Tesla T4


## The photos and the class weights

`train_d22.csv` must start with every row of `train_d17.csv`, unchanged; validation, test,
PlantDoc and the selection set are the same files v1 and v2 were scored on.

In [17]:
names = ["train_d17", "train_d22", "val", "test", "ood_test", "selection"]
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in names}
n_old = len(splits["train_d17"])
assert splits["train_d22"]["path"].iloc[:n_old].tolist() == splits["train_d17"]["path"].tolist(), \
    "train_d22.csv must start with train_d17.csv unchanged"

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
assert set(splits["train_d22"]["class"]) == set(CLASSES)

CLASS_WEIGHTS = training.balanced_class_weights(splits["train_d22"]["class"].map(INDEX), len(CLASSES), CAP)
weights = pd.DataFrame({"photos": splits["train_d22"]["class"].value_counts().reindex(CLASSES),
                        "class weight": CLASS_WEIGHTS}, index=CLASSES)

# The same numbers notebook 16 showed before the decision.
preview = pd.read_csv(ML_DIR / "results" / "d22_maize_data" / "class_balance.csv", index_col=0)
assert np.allclose(weights["class weight"], preview.loc[CLASSES, f"capped at {CAP:g} (preview)"], atol=0.006), \
    "weights differ from notebook 16's class_balance.csv"
print("Largest / smallest class:", round(weights["photos"].max() / weights["photos"].min(), 1))
display(weights.sort_values("photos", ascending=False))

Largest / smallest class: 4.0


,photos,class weight
Potato___Late_blight,2904,0.5208
Potato___Early_blight,2585,0.5850
Tomato___Late_blight,1722,0.8782
Tomato___Bacterial_spot,1629,0.9283
Tomato___healthy,1506,1.0042
Potato___healthy,1449,1.0437
Corn_(maize)___healthy,1212,1.2477
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,1094,1.3823
Corn_(maize)___Common_rust_,971,1.5574
Corn_(maize)___Northern_Leaf_Blight,844,1.7918


In [18]:
from torch.utils.data import DataLoader

LOCAL_ROOT = Path("/content/image_cache")
all_paths = sorted(set(pd.concat([splits[n] for n in names if n != "train_d17"])["path"]))
cached = image_dataset.cache_images(all_paths, config.RAW_DATA_DIR, LOCAL_ROOT)
assert cached == len(all_paths), f"Only {cached} of {len(all_paths)} images cached"


def loader(rows, train):
    transform = image_dataset.train_transform(RECIPE["image_size"]) if train else image_dataset.eval_transform(RECIPE["image_size"])
    dataset = image_dataset.SplitDataset(rows, LOCAL_ROOT, CLASSES, transform)
    return DataLoader(dataset, batch_size=RECIPE["batch_size"], shuffle=train, drop_last=train,
                      num_workers=2, pin_memory=True, persistent_workers=True)


EVAL_SETS = ["val", "test", "ood_test", "selection"]
EVAL = {name: loader(splits[name], False) for name in EVAL_SETS}
TRAIN_LOADER = loader(splits["train_d22"], True)
print("Cached", cached, "photos")

Caching images:   0%|          | 0/26235 [00:00<?, ?it/s]

Cached 26235 photos


## Train and score

Each run is scored on the four sets as soon as it finishes; the raw scores go to
`d22_comparison/` for notebook 18. Finished runs are skipped; an interrupted run resumes.

In [19]:
def train(run_name, architecture, seed, learning_rate):
    run_dir = config.MODELS_DIR / run_name
    if (run_dir / "model.pt").exists():
        print(run_name, "already trained: skipped")
        return run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    settings = {**RECIPE, "learning_rate": learning_rate,
                "architecture": f"{architecture} (ImageNet weights: {training.ARCHITECTURES[architecture][1].name})",
                "seed": seed, "train_list": "train_d22.csv", "class_weights": CLASS_WEIGHTS,
                "class_weight_rule": WEIGHT_RULE, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}
    training.save_json(settings, run_dir / "settings.json")
    training.seed_everything(seed)
    model = training.build_model(len(CLASSES), architecture)
    training.fit(model, TRAIN_LOADER, EVAL["val"], run_dir, settings, DEVICE)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    training.export(model.cpu(), CLASSES, RECIPE["image_size"], image_dataset.MEAN, image_dataset.STD,
                    run_dir / "model.pt", settings, architecture=architecture)
    (run_dir / "last.pt").unlink(missing_ok=True)   # only needed to resume; frees Drive space
    del model
    torch.cuda.empty_cache()
    return run_dir


def scores_for(run_name, run_dir, architecture):
    saved = {name: OUT_DIR / f"{run_name}__{name}.npy" for name in EVAL_SETS}
    if all(p.exists() for p in saved.values()):
        return {name: np.load(p) for name, p in saved.items()}
    model = training.build_model(len(CLASSES), architecture, pretrained=False)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    model.to(DEVICE)
    out = {}
    for name in EVAL_SETS:
        out[name], _ = training.predict(model, EVAL[name], DEVICE)
        np.save(saved[name], out[name])
    del model
    torch.cuda.empty_cache()
    return out


LOGITS = {}
for name, (architecture, seed, learning_rate) in RUN_PLAN.items():
    run_dir = train(name, architecture, seed, learning_rate)
    LOGITS[name] = scores_for(name, run_dir, architecture)
    print(name, "scored")

d22_resnet50_seed42 already trained: skipped
d22_resnet50_seed42 scored
d22_resnet50_seed43 already trained: skipped
d22_resnet50_seed43 scored
d22_resnet50_seed44 already trained: skipped
d22_resnet50_seed44 scored
d22_efficientnet_b0 already trained: skipped
d22_efficientnet_b0 scored
d22_mobilenet_v3_large already trained: skipped
d22_mobilenet_v3_large scored
d22_densenet121 already trained: skipped
d22_densenet121 scored
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 171MB/s]


Training:   0%|          | 0/259 [00:00<?, ?it/s]

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 1/15: train loss 0.6098 | val loss 0.2981 | val acc 0.9002 | val macro-F1 0.8955  (best) | 2.2 min


...

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Predicting:   0%|          | 0/39 [00:00<?, ?it/s]

Predicting:   0%|          | 0/15 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

d22_swin_t scored


## First look (the decision is notebook 18's)

Every new run next to v2, on the same photos. Selection-set accuracy is with the crop given,
as the app works; PlantDoc is reported only.

In [20]:
LOGITS["v2 (" + V2_RUN + ")"] = {name: np.load(V2_SCORES / f"{V2_RUN}__{name}.npy") for name in EVAL_SETS}
LABELS = {name: splits[name]["class"].map(INDEX) for name in EVAL_SETS}


def crop_given_correct(logits, name):
    labels = LABELS[name]
    keep = labels.notna().to_numpy()
    lab = labels[keep].astype(int).to_numpy()
    _, predicted = cal.crop_confidence(metrics.softmax(logits[keep]), CLASS_CROPS, CLASS_CROPS[lab])
    return predicted == lab


FIELD = splits["val"]["source"].isin(["fieldplant", "tanzania"]).to_numpy()
val_lab = LABELS["val"].astype(int).to_numpy()
test_lab = LABELS["test"].astype(int).to_numpy()
rows = []
for run, out in LOGITS.items():
    history = config.MODELS_DIR / run / "history.csv"
    rows.append({
        "run": run,
        "val macro-F1": metrics.summarise(out["val"], val_lab, len(CLASSES))["macro_f1"],
        "val field macro-F1": metrics.summarise(out["val"][FIELD], val_lab[FIELD], len(CLASSES))["macro_f1"],
        "test macro-F1": metrics.summarise(out["test"], test_lab, len(CLASSES))["macro_f1"],
        "selection accuracy (crop given)": crop_given_correct(out["selection"], "selection").mean(),
        "PlantDoc accuracy (crop given, reported only)": crop_given_correct(out["ood_test"], "ood_test").mean(),
        "training minutes": pd.read_csv(history)["minutes"].sum() if history.exists() and run in RUN_PLAN else np.nan,
    })
table = pd.DataFrame(rows).set_index("run")
display(table.round(4))

,val macro-F1,val field macro-F1,test macro-F1,selection accuracy (crop given),"PlantDoc accuracy (crop given, reported only)",training minutes
run,,,,,,
d22_resnet50_seed42,0.9841,0.9938,0.9844,0.6548,0.5257,29.0160
d22_resnet50_seed43,0.9859,0.9689,0.9872,0.6689,0.5551,28.8870
d22_resnet50_seed44,0.9830,0.9621,0.9908,0.6726,0.5142,28.7356
d22_efficientnet_b0,0.9876,0.9812,0.9901,0.7145,0.5373,26.3225
d22_mobilenet_v3_large,0.9859,0.9791,0.9887,0.6880,0.5404,24.9200
d22_densenet121,0.9856,0.9597,0.9880,0.6276,0.5446,28.4564
d22_convnext_tiny,0.9865,0.9559,0.9893,0.6854,0.5110,29.5303
d22_convnext_tiny_lr1e-04,0.9880,0.9628,0.9877,0.7242,0.5887,29.7501
d22_swin_t,0.9880,0.9450,0.9892,0.6205,0.5121,33.0191


## Save the results

In [21]:
table.to_csv(OUT_DIR / "runs.csv")
weights.to_csv(OUT_DIR / "class_weights.csv")

REPO_RESULTS = ML_DIR / "results" / "d22_training"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in [OUT_DIR / "runs.csv", OUT_DIR / "class_weights.csv"]:
    shutil.copy(file, REPO_RESULTS / file.name)
for run in RUN_PLAN:
    for file in ["settings.json", "history.csv"]:
        shutil.copy(config.MODELS_DIR / run / file, REPO_RESULTS / f"{run}__{file}")
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

['class_weights.csv', 'd22_convnext_tiny__history.csv', 'd22_convnext_tiny__settings.json', 'd22_convnext_tiny_lr1e-04__history.csv', 'd22_convnext_tiny_lr1e-04__settings.json', 'd22_densenet121__history.csv', 'd22_densenet121__settings.json', 'd22_efficientnet_b0__history.csv', 'd22_efficientnet_b0__settings.json', 'd22_mobilenet_v3_large__history.csv', 'd22_mobilenet_v3_large__settings.json', 'd22_resnet50_seed42__history.csv', 'd22_resnet50_seed42__settings.json', 'd22_resnet50_seed43__history.csv', 'd22_resnet50_seed43__settings.json', 'd22_resnet50_seed44__history.csv', 'd22_resnet50_seed44__settings.json', 'd22_swin_t__history.csv', 'd22_swin_t__settings.json', 'notebook_resnet.ipynb', 'runs.csv']


In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

Mounted at /content/drive
Drive synced and remounted.


## Save to GitHub

Save the notebook first (Ctrl+S); only the results folder is committed, to a new branch
named after the runs it holds (`resnet` after the first three, `all` at the end), so a
second push never clashes with the first.

In [ ]:
from google.colab import userdata

from src import github_sync

PART = "all" if TRAIN_OTHERS else "resnet"
shutil.copy(ML_DIR / "notebooks" / "17_d22_training.ipynb", REPO_RESULTS / f"notebook_{PART}.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch=f"colab/d22-training-{PART}",
    message=f"D22: train with class weights on train_d22.csv ({len(RUN_PLAN)} runs)",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))